# dlcobra エラー分析 (Top-K Error Analysis)

モデルの予測エラーを分析する。
- 高確信度の誤り局面を抽出
- 将棋盤SVGで可視化
- スライス別（序盤/中盤/終盤）の精度確認

In [ ]:
import sys
import numpy as np
import torch
import cshogi
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

DEVICE = 'cpu'  # GPU学習中のためCPUを使用
MAX_SAMPLES = 200  # CPUなので少なめに
print(f'Device: {DEVICE}')

## 1. モデルのロード

In [ ]:
from dlshogi.experiments.exp026_droppath_bf16_fresh.model import PolicyValueNetwork

# exp026 checkpoint（現在学習中のrun: bz5nhn8j）
# 学習が進んだら新しいstepのものに変更
CKPT_PATH = '../dlshogi/wandb/wcsc36/bz5nhn8j/checkpoints/last.ckpt'

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
# torch.compile使用時はキーが '_orig_mod.' プレフィックスになる
state_dict = {k.replace('model._orig_mod.', '', 1): v 
              for k, v in ckpt['state_dict'].items() 
              if k.startswith('model._orig_mod.')}
if not state_dict:  # compile未使用の場合のフォールバック
    state_dict = {k.replace('model.', '', 1): v 
                  for k, v in ckpt['state_dict'].items() 
                  if k.startswith('model.')}

model = PolicyValueNetwork().to(DEVICE)
model.load_state_dict(state_dict)
model.eval()
print('Loaded:', CKPT_PATH)
print('step:', ckpt.get('global_step', 'unknown'))

## 2. SFENつきエラー収集

In [ ]:
from cshogi.dlshogi import make_input_features, make_move_label, FEATURES1_NUM, FEATURES2_NUM
import numpy as np

VAL_FILE = '/mnt/nvme1n1p2/data/shogi-ai-book/floodgate_test_2017-2018_r3500_eval5000.hcpe'
MAX_SAMPLES = 200  # CPUの場合は少なめに

hcpes = np.fromfile(VAL_FILE, dtype=cshogi.HuffmanCodedPosAndEval)
board = cshogi.Board()
sfen_errors = []
total = 0

for hcpe in hcpes[:MAX_SAMPLES]:
    board.set_hcp(hcpe['hcp'])
    true_move16 = hcpe['bestMove16']
    true_label = make_move_label(true_move16, board.turn)
    total += 1

    features1 = np.empty((FEATURES1_NUM, 9, 9), dtype=np.float32)
    features2 = np.empty((FEATURES2_NUM, 9, 9), dtype=np.float32)
    make_input_features(board, features1, features2)
    x1 = torch.tensor(features1).unsqueeze(0).to(DEVICE)
    x2 = torch.tensor(features2).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        policy, _ = model(x1, x2)
        prob = torch.softmax(policy, dim=1)
        pred_label = prob.argmax().item()
        confidence = prob.max().item()

    if pred_label != true_label:
        sfen_errors.append({
            'sfen': board.sfen(),
            'true_move16': true_move16,
            'pred_label': pred_label,
            'confidence': confidence,
            'move_number': board.move_number,
        })

sfen_errors.sort(key=lambda x: x['confidence'], reverse=True)
print(f'Accuracy: {1 - len(sfen_errors)/total:.4f} ({total - len(sfen_errors)}/{total})')
print(f'Errors: {len(sfen_errors)}')

## 3. 局面の可視化（高確信度の誤り）

In [ ]:
from IPython.display import display, SVG

def show_error(err, idx):
    b = cshogi.Board(err['sfen'])
    print(f"[{idx}] 手数:{err['move_number']}  確信度:{err['confidence']:.3f}  正解手:{cshogi.move_to_usi(err['true_move16'])}")
    display(b.to_svg(lastmove=err['true_move16'], scale=0.7))

TOP_N = 10  # 表示件数
for i, err in enumerate(sfen_errors[:TOP_N]):
    show_error(err, i)

## 4. スライス分析（序盤/中盤/終盤）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'IPAexGothic'  # 日本語フォント（なければ英語表記に変更）

def phase(move_number):
    if move_number <= 40: return 'Opening'
    elif move_number <= 80: return 'Middlegame'
    else: return 'Endgame'

phase_stats = {'Opening': [0, 0], 'Middlegame': [0, 0], 'Endgame': [0, 0]}  # [errors, total]

for hcpe in hcpes[:MAX_SAMPLES]:
    board.set_hcp(hcpe['hcp'])
    phase_stats[phase(board.move_number)][1] += 1

for err in sfen_errors:
    phase_stats[phase(err['move_number'])][0] += 1

phases = list(phase_stats.keys())
accs = [(t - e) / t if t > 0 else 0 for e, t in phase_stats.values()]

plt.figure(figsize=(6, 4))
bars = plt.bar(phases, accs, color=['#4CAF50', '#2196F3', '#FF9800'])
plt.ylim(0, 1)
plt.ylabel('Policy Accuracy')
plt.title('Policy Accuracy by Game Phase')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.3f}', ha='center')
plt.tight_layout()
plt.show()

for p, (e, t) in zip(phases, phase_stats.values()):
    print(f'{p}: acc={(t-e)/t:.4f}  errors={e}/{t}')